In [1]:
#!/usr/bin/env python
from __future__ import annotations

import argparse
import logging
import math
import os
from functools import partial

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)

from retnet_hf import RetNetConfig, RetNetForCausalLM

logger = logging.getLogger(__name__)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train a TorchScale RetNet wrapped as a Hugging Face CausalLM.")
    parser.add_argument("--dataset_name", type=str, default="FiscalNote/billsum")
    parser.add_argument("--dataset_config_name", type=str, default=None)
    parser.add_argument("--dataset_text_field", type=str, default=None)
    parser.add_argument("--streaming", action="store_true")
    parser.add_argument("--tokenizer_name", type=str, default="gpt2")
    parser.add_argument("--output_dir", type=str, default="outputs/retnet-billsum")
    parser.add_argument("--block_size", type=int, default=512)
    parser.add_argument("--max_train_samples", type=int, default=None)
    parser.add_argument("--max_eval_samples", type=int, default=1025)
    parser.add_argument("--validation_split_percentage", type=int, default=5)
    parser.add_argument("--per_device_train_batch_size", type=int, default=2)
    parser.add_argument("--per_device_eval_batch_size", type=int, default=2)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=8)
    parser.add_argument("--learning_rate", type=float, default=3e-4)
    parser.add_argument("--weight_decay", type=float, default=0.01)
    parser.add_argument("--num_train_epochs", type=float, default=1.0)
    parser.add_argument("--max_steps", type=int, default=-1)
    parser.add_argument("--warmup_ratio", type=float, default=0.03)
    parser.add_argument("--logging_steps", type=int, default=20)
    parser.add_argument("--save_steps", type=int, default=200)
    parser.add_argument("--eval_steps", type=int, default=200)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--bf16", action="store_true")
    parser.add_argument("--fp16", action="store_true")
    parser.add_argument("--push_to_hub", action="store_true")
    parser.add_argument("--hub_model_id", type=str, default=None)
    parser.add_argument("--hidden_size", type=int, default=512)
    parser.add_argument("--intermediate_size", type=int, default=2048)
    parser.add_argument("--num_hidden_layers", type=int, default=8)
    parser.add_argument("--num_retention_heads", type=int, default=8)
    parser.add_argument("--value_dim", type=int, default=None)
    parser.add_argument("--dropout", type=float, default=0.1)
    parser.add_argument("--activation_dropout", type=float, default=0.0)
    parser.add_argument("--drop_path_rate", type=float, default=0.0)
    parser.add_argument("--recurrent_chunk_size", type=int, default=128)
    parser.add_argument("--chunkwise_recurrent", action="store_true")
    parser.add_argument("-f", type=str, default="does nothing")
    return parser.parse_args()


def infer_text_field(dataset_name: str, dataset_text_field: str | None) -> str | None:
    if dataset_text_field:
        return dataset_text_field
    lower = dataset_name.lower()
    if "billsum" in lower:
        return None  # handled by formatter below
    if "c4" in lower:
        return "text"
    return "text"


def load_raw_datasets(args: argparse.Namespace):
    lower = args.dataset_name.lower()
    if "billsum" in lower:
        ds = load_dataset(args.dataset_name, args.dataset_config_name)
        if "validation" in ds:
            train_ds, eval_ds = ds["train"], ds["validation"]
        else:
            if args.streaming:
                raise ValueError("Streaming Billsum validation split creation is not supported in this example.")
            split = ds["train"].train_test_split(
                test_size=args.validation_split_percentage / 100.0,
                seed=args.seed,
            )
            train_ds, eval_ds = split["train"], split["test"]
        return train_ds, eval_ds

    if "c4" in lower:
        config_name = args.dataset_config_name or "en"
        train_ds = load_dataset(args.dataset_name, config_name, split="train", streaming=args.streaming)
        eval_ds = load_dataset(args.dataset_name, config_name, split="validation", streaming=args.streaming)
        return train_ds, eval_ds

    ds = load_dataset(args.dataset_name, args.dataset_config_name)
    if "validation" in ds:
        return ds["train"], ds["validation"]
    if args.streaming:
        raise ValueError("Streaming is only wired here for datasets that already expose validation splits.")
    split = ds["train"].train_test_split(
        test_size=args.validation_split_percentage / 100.0,
        seed=args.seed,
    )
    return split["train"], split["test"]


def format_for_causal_lm(example, dataset_name: str, text_field: str | None):
    lower = dataset_name.lower()
    if "billsum" in lower:
        return {
            "text": f"Document:\n{example['text']}\n\nSummary:\n{example['summary']}"
        }
    if text_field is None:
        raise ValueError("Could not infer dataset text field. Pass --dataset_text_field.")
    return {"text": example[text_field]}


def tokenize_function(examples, tokenizer):
    return tokenizer(examples["text"])


def group_texts(examples, block_size: int):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples["input_ids"])
    total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = list(result["input_ids"])
    return result

In [2]:
def main() -> None:
    args = parse_args()
    logging.basicConfig(
        format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
        level=logging.INFO,
    )
    set_seed(args.seed)

    text_field = infer_text_field(args.dataset_name, args.dataset_text_field)
    train_raw, eval_raw = load_raw_datasets(args)

    tokenizer = AutoTokenizer.from_pretrained(args.tokenizer_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    format_fn = partial(format_for_causal_lm, dataset_name=args.dataset_name, text_field=text_field)

    remove_columns = getattr(train_raw, "column_names", None)
    train_ds = train_raw.map(format_fn, remove_columns=remove_columns)
    eval_ds = eval_raw.map(format_fn, remove_columns=getattr(eval_raw, "column_names", None))

    tokenize_fn = partial(tokenize_function, tokenizer=tokenizer)
    train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
    eval_ds = eval_ds.map(tokenize_fn, batched=True, remove_columns=["text"])

    if args.max_train_samples is not None and not args.streaming:
        train_ds = train_ds.select(range(min(args.max_train_samples, len(train_ds))))
    if args.max_eval_samples is not None and not args.streaming:
        eval_ds = eval_ds.select(range(min(args.max_eval_samples, len(eval_ds))))

    group_fn = partial(group_texts, block_size=args.block_size)
    train_ds = train_ds.map(group_fn, batched=True)
    eval_ds = eval_ds.map(group_fn, batched=True)

    config = RetNetConfig(
        vocab_size=len(tokenizer),
        hidden_size=args.hidden_size,
        intermediate_size=args.intermediate_size,
        num_hidden_layers=args.num_hidden_layers,
        num_retention_heads=args.num_retention_heads,
        value_dim=args.value_dim,
        hidden_dropout_prob=args.dropout,
        activation_dropout=args.activation_dropout,
        drop_path_rate=args.drop_path_rate,
        recurrent_chunk_size=args.recurrent_chunk_size,
        chunkwise_recurrent=args.chunkwise_recurrent,
        pad_token_id=tokenizer.pad_token_id,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    model = RetNetForCausalLM(config)
    model.resize_token_embeddings(len(tokenizer))

    training_args = TrainingArguments(
        output_dir=args.output_dir,
        overwrite_output_dir=True,
        do_train=True,
        do_eval=True,
        evaluation_strategy="steps",
        eval_steps=args.eval_steps,
        save_steps=args.save_steps,
        logging_steps=args.logging_steps,
        per_device_train_batch_size=args.per_device_train_batch_size,
        per_device_eval_batch_size=args.per_device_eval_batch_size,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        learning_rate=args.learning_rate,
        weight_decay=args.weight_decay,
        num_train_epochs=args.num_train_epochs,
        max_steps=args.max_steps,
        warmup_ratio=args.warmup_ratio,
        lr_scheduler_type="cosine",
        bf16=args.bf16,
        fp16=args.fp16,
        push_to_hub=args.push_to_hub,
        hub_model_id=args.hub_model_id,
        remove_unused_columns=False,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        tokenizer=tokenizer,
        data_collator=default_data_collator,
    )

    train_result = trainer.train()
    trainer.save_model(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)

    metrics = trainer.evaluate()
    if "eval_loss" in metrics:
        metrics["eval_perplexity"] = math.exp(metrics["eval_loss"])

    trainer.log_metrics("train", train_result.metrics)
    trainer.save_metrics("train", train_result.metrics)
    trainer.log_metrics("eval", metrics)
    trainer.save_metrics("eval", metrics)
    trainer.save_state()

    print("Saved model to", os.path.abspath(args.output_dir))
    print("Eval metrics:", metrics)


#if __name__ == "__main__":
#    main()


In [3]:
args = parse_args()
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    level=logging.INFO,
)
set_seed(args.seed)

In [4]:
text_field = infer_text_field(args.dataset_name, args.dataset_text_field)
train_raw, eval_raw = load_raw_datasets(args)

train_raw = train_raw.select(range(min(50, len(train_raw))))
eval_raw = eval_raw.select(range(min(20, len(train_raw))))

tokenizer = AutoTokenizer.from_pretrained(args.tokenizer_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

format_fn = partial(format_for_causal_lm, dataset_name=args.dataset_name, text_field=text_field)

remove_columns = getattr(train_raw, "column_names", None)
train_ds = train_raw.map(format_fn, remove_columns=remove_columns)
eval_ds = eval_raw.map(format_fn, remove_columns=getattr(eval_raw, "column_names", None))

tokenize_fn = partial(tokenize_function, tokenizer=tokenizer)
train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
eval_ds = eval_ds.map(tokenize_fn, batched=True, remove_columns=["text"])

if args.max_train_samples is not None and not args.streaming:
    train_ds = train_ds.select(range(min(args.max_train_samples, len(train_ds))))
if args.max_eval_samples is not None and not args.streaming:
    eval_ds = eval_ds.select(range(min(args.max_eval_samples, len(eval_ds))))

group_fn = partial(group_texts, block_size=args.block_size)
train_ds = train_ds.map(group_fn, batched=True)
eval_ds = eval_ds.map(group_fn, batched=True)

2026-05-01 21:42:23,256 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/datasets/FiscalNote/billsum "HTTP/1.1 200 OK"
2026-05-01 21:42:23,374 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/datasets/FiscalNote/billsum "HTTP/1.1 200 OK"
2026-05-01 21:42:23,649 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/datasets/FiscalNote/billsum/revision/3d8510441c06a3d9dfb32eb0d7f80151730bcc4f "HTTP/1.1 200 OK"
2026-05-01 21:42:23,674 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/datasets/FiscalNote/billsum/tree/3d8510441c06a3d9dfb32eb0d7f80151730bcc4f/data?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-05-01 21:42:23,699 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/datasets/FiscalNote/billsum/tree/3d8510441c06a3d9dfb32eb0d7f80151730bcc4f?recursive=false&expand=false "HTTP/1.1 200 OK"
2026-05-01 21:42:24,126 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
2026-

In [5]:
config = RetNetConfig(
    vocab_size=len(tokenizer),
    hidden_size=args.hidden_size,
    intermediate_size=args.intermediate_size,
    num_hidden_layers=args.num_hidden_layers,
    num_retention_heads=args.num_retention_heads,
    value_dim=args.value_dim,
    hidden_dropout_prob=args.dropout,
    activation_dropout=args.activation_dropout,
    drop_path_rate=args.drop_path_rate,
    recurrent_chunk_size=args.recurrent_chunk_size,
    chunkwise_recurrent=args.chunkwise_recurrent,
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
model = RetNetForCausalLM(config)
model.resize_token_embeddings(len(tokenizer))

Embedding(50257, 512)

In [6]:
print (model._tied_weights_keys)

{'model.decoder.embed_tokens.weight': 'model.embed_tokens.weight', 'model.decoder.output_projection.weight': 'model.embed_tokens.weight', 'lm_head.weight': 'model.decoder.output_projection.weight'}


In [7]:
training_args = TrainingArguments(
    output_dir=args.output_dir,
    #overwrite_output_dir=True,
    do_train=True,
    do_eval=True,
    eval_strategy ="steps",
    eval_steps=args.eval_steps,
    save_steps=args.save_steps,
    logging_steps=args.logging_steps,
    per_device_train_batch_size=args.per_device_train_batch_size,
    per_device_eval_batch_size=args.per_device_eval_batch_size,
    gradient_accumulation_steps=args.gradient_accumulation_steps,
    learning_rate=args.learning_rate,
    weight_decay=args.weight_decay,
    num_train_epochs=args.num_train_epochs,
    max_steps=args.max_steps,
    warmup_ratio=args.warmup_ratio,
    lr_scheduler_type="cosine",
    bf16=args.bf16,
    fp16=args.fp16,
    push_to_hub=args.push_to_hub,
    hub_model_id=args.hub_model_id,
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=default_data_collator,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [8]:
train_result = trainer.train()
trainer.save_model(args.output_dir)
tokenizer.save_pretrained(args.output_dir)

metrics = trainer.evaluate()
if "eval_loss" in metrics:
    metrics["eval_perplexity"] = math.exp(metrics["eval_loss"])

trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.log_metrics("eval", metrics)
trainer.save_metrics("eval", metrics)
trainer.save_state()

print("Saved model to", os.path.abspath(args.output_dir))
print("Eval metrics:", metrics)

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
22,48.607095,5.245567


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step
48.607095,5.245567,22


***** train metrics *****
  epoch                    =        1.0
  total_flos               =    35912GF
  train_loss               =    47.3779
  train_runtime            = 0:00:15.37
  train_samples_per_second =     22.889
  train_steps_per_second   =      1.431
***** eval metrics *****
  eval_loss       =   5.2456
  eval_perplexity = 189.7233
Saved model to C:\Users\Za\Desktop\SchoolWork\GMU\757 GAN\retnet_hf_wrapper\scripts\outputs\retnet-billsum
Eval metrics: {'eval_loss': 5.2455668449401855, 'eval_perplexity': 189.72332846374934}


In [9]:

# Prompt (match your training format!)
prompt = """Document:
The Crusading movement was a major religious, political, and military endeavour of the Middle Ages, generally dated from the Council of Clermont (1095), at which Pope Urban II proclaimed an armed expedition in support of Eastern Christians under Muslim rule. He framed it as a form of penitential pilgrimage. By this point, papal authority had grown through church reforms, and tensions with secular rulers encouraged the notion of holy war—combining classical just war theory, biblical precedents, and Augustine's teachings on legitimate violence. Armed pilgrimage aligned with the era's Christocentric and militant Catholicism, sparking widespread enthusiasm. Western expansion was further enabled by economic growth, the decline of older Mediterranean powers, and Muslim disunity. These factors allowed crusaders to seize territory and found four Crusader states in the Levant, whose defence inspired successive Crusades. The papacy also launched crusading campaigns against other targets—Muslims in Iberia, Paganism in the Baltic, and other opponents of papal authority.

Though aimed primarily at the warrior elite through appeals to chivalric ideals, the movement depended on broad support from clergy, townspeople, and peasants. Women, despite being discouraged, were involved as participants, proxies for absent crusaders, or victims. Although many crusaders were motivated by indulgences (absolution from sins), material gain also played a part. Crusading campaigns were typically initiated through papal bulls, and participants pledged to join by "taking the cross"—sewing a cross onto their garments. Failure to fulfil vows could result in excommunication. Periodic waves of zeal produced unsanctioned "popular crusades".

The papal-sanctioned wars fostered distinctive institutions and ideologies. Initially funded through improvised means, later campaigns received more organized support via papal taxes on clergy and the sale of indulgences. Core crusading forces were heavily armed knights, backed by infantry, local troops, and naval aid from maritime cities. Crusaders secured their holdings by building strong castles, and the fusion of chivalric and monastic ideals led to the rise of military orders. The movement extended Western Christendom and created new frontier states, some surviving into the early modern period. Crusading encouraged cultural exchange and left lasting marks on European art and literature. Despite a decline during the Reformation, anti-Ottoman "holy leagues" sustained the tradition into the 18th century. 

Summary:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,       # or False for greedy
        temperature=0.7,
        top_p=0.9,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

TypeError: 'DynamicCache' object does not support item assignment